# Bike sharing demand

Predict hourly rental count from weather and calendar features.
UCI Bike Sharing, Washington D.C., 2011-2012.

1. read the raw CSV from S3
2. train a regressor on this instance
3. write the model back to S3

This is a **regression** problem: the target is a count, not a class.

## 1. Setup

`conda_python3` is a minimal kernel, so the libraries are installed first.

In [ ]:
%pip install -q scikit-learn pandas

In [ ]:
import boto3

REGION = "ca-central-1"

# From `terraform -chdir=infra/mlops output data_bucket`. The suffix is
# random, so it changes on every destroy/apply cycle.
BUCKET = "sagemaker-notebook-dev-data-3opmm6"

RAW_KEY = "raw/bike/hour.csv"
MODEL_KEY = "models/bike/model.joblib"

s3 = boto3.client("s3", region_name=REGION)

s3.head_bucket(Bucket=BUCKET)
print(f"bucket reachable: {BUCKET}")

## 2. Load

17,379 hourly records. `cnt` is the target.

In [ ]:
import io

import pandas as pd

obj = s3.get_object(Bucket=BUCKET, Key=RAW_KEY)
df = pd.read_csv(io.BytesIO(obj["Body"].read()))

print(f"{len(df)} rows, {df.dteday.min()} -> {df.dteday.max()}")
df.head()

## 3. Features

`casual` and `registered` sum to `cnt` in every row, so keeping them
would leak the target and produce a meaningless perfect score. `instant`
is a row index and `dteday` is superseded by the calendar columns.

In [ ]:
TARGET = "cnt"
DROP = ["instant", "dteday", "casual", "registered", TARGET]
FEATURES = [c for c in df.columns if c not in DROP]

# Proof of the leak, worth seeing once.
leak = (df.casual + df.registered != df[TARGET]).sum()
print(f"rows where casual + registered != cnt: {leak}")
print(f"\
{len(FEATURES)} features: {FEATURES}")

## 4. Split by time

Train on 2011, test on 2012. A random split would let the model see
hours adjacent to the ones it is scored on, which inflates the result.

Note `yr` is constant within each split, so the model cannot use it. It
stays in the feature list to match the CSV schema.

In [ ]:
train = df[df.yr == 0]
test = df[df.yr == 1]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")

## 5. Train

Random forest handles the mix of categorical codes and normalized
floats without scaling or encoding.

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"rmse={rmse:.1f}  mae={mean_absolute_error(y_test, pred):.1f}  "
      f"r2={r2_score(y_test, pred):.4f}")
print(f"(test mean count = {y_test.mean():.0f})")

## 6. What drives demand

In [ ]:
importance = sorted(zip(FEATURES, model.feature_importances_), key=lambda x: -x[1])

for name, score in importance[:6]:
    print(f"  {name:12} {score:.3f}  {'#' * int(score * 60)}")

## 7. Save the model to S3

In [ ]:
import joblib

buf = io.BytesIO()
joblib.dump(model, buf)
buf.seek(0)

s3.upload_fileobj(buf, BUCKET, MODEL_KEY)
print(f"s3://{BUCKET}/{MODEL_KEY}")

## 8. Verify the round trip

In [ ]:
obj = s3.get_object(Bucket=BUCKET, Key=MODEL_KEY)
reloaded = joblib.load(io.BytesIO(obj["Body"].read()))

for p, a in zip(reloaded.predict(X_test.head(5)), y_test.head(5)):
    print(f"  predicted={p:7.1f}  actual={a:5d}")